# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sarahsaeed564/Machine_Learning/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**Answer:** Classification. The decision I named in ML-02 is "which pages should a reviewer check first," but the data already ships with an observed, measured outcome for each page — `trend_direction`, computed from real impressions in the last 30 days vs. the prior 30 days. That makes "will this page decline?" a yes/no classification problem, not an open-ended ranking problem with no ground truth. Once a model outputs a *probability* of decline for each page, that probability doubles as the priority score for the reviewer — so classification is the task type, and ranking-by-score is how the output gets used.

In [33]:
!ls /content

Machine_Learning  sample_data


In [34]:
!git clone https://github.com/sarahsaeed564/Machine_Learning.git

Cloning into 'Machine_Learning'...
remote: Enumerating objects: 139, done.
remote: Counting objects: 100% (139/139), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 139 (delta 52), reused 103 (delta 34), pack-reused 0 (from 0)
Receiving objects: 100% (139/139), 1.85 MiB | 15.64 MiB/s, done.
Resolving deltas: 100% (52/52), done.


In [35]:
%cd /content/Machine_Learning

/content/Machine_Learning


In [36]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Task type check — trend_direction values:")
print(df["trend_direction"].value_counts())


Task type check — trend_direction values:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Answer:** Target: `is_declining_label`, defined as `1` if `trend_direction == "down"` else `0`. This comes from an **observed outcome** — the actual change in a page's impressions over the last 30 days vs. the previous 30 days — not from someone hand-labeling pages. The one thing to be careful about: `trend_direction` and the `trend_pct` it's computed from are the label's own source, so they can never be used as input features later — that would be leaking the answer into the inputs.


In [37]:
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Positive class rate (declining pages):", round(df["is_declining_label"].mean() * 100, 1), "%")
print("Declining pages:", df["is_declining_label"].sum(), "out of", len(df))

Positive class rate (declining pages): 54.2 %
Declining pages: 16262 out of 30000


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Answer: precision@K**, where K = however many pages a reviewer can realistically check in a week (say the top 500 by predicted decline probability). This ties the metric directly to the real decision: of the pages the model sends to a reviewer, what share are actually declining? ROC-AUC is a useful secondary sanity check since the classes are fairly balanced (54% vs 46%), but precision@K is what I'd defend to a reviewer, because it answers "was my time well spent?" — which is the actual cost named in ML-02.

In [38]:
# Baseline precision@K if we just grabbed pages at random (no model) —
# this is the number any real model has to beat.
K = 500
baseline_precision_at_k = df["is_declining_label"].mean()  # random pick ≈ overall decline rate
print(f"Baseline precision@{K} (random selection):", round(baseline_precision_at_k * 100, 1), "%")

Baseline precision@500 (random selection): 54.2 %


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Answer:** One row = one content page (one pseudonymized page belonging to one client). Below is a real slice of the dataframe showing the identifiers, a few decision-relevant signals, and the target column.

In [39]:
unit_cols = ["content_id", "client_id", "content_type", "impressions_90d",
             "search_volume", "avg_position", "freshness_tier", "trend_direction", "is_declining_label"]

print("rows, cols:", df.shape)
df[unit_cols].head(5)


rows, cols: (30000, 45)


,content_id,client_id,content_type,impressions_90d,search_volume,avg_position,freshness_tier,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,3803,10.0,10.6,0-30,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,15320,90.0,20.3,0-30,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,12581,0.0,36.5,0-30,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,11751,10.0,6.2,0-30,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,19140,0.0,44.0,0-30,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

**Answer:** A fixed rule already exists — that's literally how `trend_direction` itself is computed (a >20% drop in impressions). But that rule can only tell you a page declined *after* the fact; it can't tell you *ahead of time* which of the 16,000+ declining pages to fix first, or which healthy-looking pages are about to slip. The signals that would predict this don't move together in a simple way — decline rate swings from 28.7% to 57.2% across content types, and it's actually *non-monotonic* across both freshness and traffic tiers (see below): the middling tiers decline more than either the freshest/highest-traffic or the stalest/lowest-traffic pages. No single if-statement threshold captures a pattern that goes up, then down, then up again across a variable. A model can weigh several such signals together instead of picking one and hoping.

In [40]:
print("Decline rate by content_type (%):")
print((df.groupby("content_type")["is_declining_label"].mean() * 100).round(1))
print()
print("Decline rate by freshness_tier (%) — notice it's NOT a straight line:")
print((df.groupby("freshness_tier")["is_declining_label"].mean() * 100).round(1))
print()
print("Decline rate by impression_tier (%) — also non-monotonic:")
print((df.groupby("impression_tier")["is_declining_label"].mean() * 100).round(1))


Decline rate by content_type (%):
content_type
comparison article    57.2
feedly article        28.7
keyword article       56.1
Name: is_declining_label, dtype: float64

Decline rate by freshness_tier (%) — notice it's NOT a straight line:
freshness_tier
0-30      51.1
181+      47.1
31-90     58.9
91-180    61.1
Name: is_declining_label, dtype: float64

Decline rate by impression_tier (%) — also non-monotonic:
impression_tier
excellent    46.2
good         58.6
low          45.4
moderate     61.5
Name: is_declining_label, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.